# Intro til Machine Learning — 5: MNIST — netværket der kan læse 🔢👁️

Finalen! I skal træne et neuralt netværk til at genkende **håndskrevne tal** — rigtige
billeder, skrevet af rigtige (amerikanske) mennesker. Datasættet hedder **MNIST** og er
ML-verdenens svar på "Hello World": 70.000 små gråtonebilleder af cifrene 0–9.

Og her er dagens vigtigste pointe, før vi overhovedet starter: **et billede er bare tal i
en tabel.** Hele jeres værktøjskasse — pandas, standardisering, tensorer, klasser,
træningsloops, softmax — virker derfor direkte. Ingen ny magi. Lad os læse nogle tal!

> ⭐ = til de hurtige, 🐛 = find den bevidste fejl.

## Setup

In [ ]:
!pip install -q kagglehub

In [ ]:
!wget -q -nc https://raw.githubusercontent.com/UNF-Science-Camps/KIC26/main/Intro-ML/hjaelpefunktioner.py

In [ ]:
import kagglehub
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

from hjaelpefunktioner import vis_mnist_billeder

torch.manual_seed(42)
np.random.seed(42)

# 13: Billeder er bare tal (i en DataFrame)

Vi henter MNIST fra Kaggle — **som CSV-fil**, så I kan se med egne øjne, at billeder bare
er endnu en tabel. (Indlæsningen tager et lille øjeblik — filen har 60.000 rækker.)

In [ ]:
sti = kagglehub.dataset_download("oddrationale/mnist-in-csv")

traen_df = pd.read_csv(sti + "/mnist_train.csv")
test_df = pd.read_csv(sti + "/mnist_test.csv")
print("træning:", traen_df.shape)
print("test:   ", test_df.shape)

> 🚨 **Plan B:** Hvis Kaggle driller, henter cellen nedenfor færdiglavede (mindre)
> udgaver fra vores GitHub — fjern `#`'erne og kør den, og **spring så subsample-cellen
> nedenfor over** (Plan B-filerne er allerede skåret ned).

In [ ]:
# 🚨 Plan B — kør KUN hvis Kaggle-cellen fejlede:
# traen_df = pd.read_csv("https://raw.githubusercontent.com/UNF-Science-Camps/KIC26/main/Intro-ML/data/mnist_traen_lille.csv.gz")
# test_df = pd.read_csv("https://raw.githubusercontent.com/UNF-Science-Camps/KIC26/main/Intro-ML/data/mnist_test_lille.csv.gz")
# print(traen_df.shape, test_df.shape)

**(60000, 785)** — en tabel med 60.000 rækker, én pr. billede. Og de 785 kolonner?
Den første er `label` (hvilket ciffer billedet forestiller), og resten er... pixels!
$28 \times 28 = 784$ pixels pr. billede, foldet ud i én lang række, med gråtoneværdier
fra 0 (sort) til 255 (hvid). Det er bogstaveligt talt bare en kæmpe Pokémon-tabel med
flere kolonner.

60.000 billeder er mere, end vores stakkels CPU behøver — vi klipper ned til 16.000
trænings- og 2.000 testbilleder (`sample` trækker tilfældige rækker, og `random_state`
gør det ens for alle):

In [ ]:
traen_df = traen_df.sample(n=16000, random_state=42).reset_index(drop=True)
test_df = test_df.sample(n=2000, random_state=42).reset_index(drop=True)
print("træning:", traen_df.shape, "| test:", test_df.shape)

Nu deler vi tabellen i **y** (første kolonne: cifret) og **X** (de 784 pixels) — og
ser det første billede som det, det i virkeligheden er: en bunke tal.

In [ ]:
y_traen_np = traen_df["label"].values
X_traen_np = traen_df.drop(columns=["label"]).values     # alt undtagen label-kolonnen

raekke = X_traen_np[0]
print("én række:", raekke.shape, "— min:", raekke.min(), " max:", raekke.max())
print("label:", y_traen_np[0])

In [ ]:
billede = raekke.reshape(28, 28)      # 784 tal → 28×28-gitter (reshape fra opgave 5.5!)

plt.imshow(billede, cmap="gray")
plt.title(f"label: {y_traen_np[0]}")
plt.colorbar(label="pixelværdi")
plt.show()

Der er cifret! `vis_mnist_billeder` fra hjælpefilen viser et helt grid ad gangen:

In [ ]:
vis_mnist_billeder(X_traen_np[:10], y_traen_np[:10], n=10)

## Klargøring: normalisér og konvertér

Pixelværdier på 0–255 er alt for store tal til et netværk (notebook 1-lektien!). Her er
alle features heldigvis på SAMME skala, så vi kan nøjes med at dele med 255 — så ligger
alt mellem 0 og 1. Derefter: tensorer, som altid. Bemærk `torch.long` til labels —
klassenumre skal være hele tal (CrossEntropy-reglen fra opgave 12.6!):

In [ ]:
X_traen = torch.tensor(X_traen_np / 255.0, dtype=torch.float32)
y_traen = torch.tensor(y_traen_np, dtype=torch.long)

X_test = torch.tensor(test_df.drop(columns=["label"]).values / 255.0, dtype=torch.float32)
y_test = torch.tensor(test_df["label"].values, dtype=torch.long)

print(X_traen.shape, X_traen.min().item(), "til", X_traen.max().item())

### Opgaver

##### Opgave 13.1
Vis billede nummer 0 — og skift så indekset og bladr lidt rundt (der er 16.000 at vælge
imellem). Find et tal, der er skrevet SÅ grimt, at I selv er i tvivl om, hvad det er.
Hvad siger labelen? Ville I have gættet rigtigt?

In [ ]:
# alt muligt er rigtigt her — fx viser et lille grid mange kandidater på én gang:
vis_mnist_billeder(X_traen_np[40:60], y_traen_np[40:60], n=20)

<span style="color:red"><b>LØSNINGSFORSLAG:</b> Der er MASSER af grimme tal i MNIST — mennesker er uenige om 1-2 % af billederne. Pointen: hvis mennesker ikke kan blive enige, kan modellen heller ikke blive perfekt. Fejlgrænsen ligger i DATAENE, ikke kun i modellen.</span>

##### Opgave 13.2
Udfyld `reshape`, så én række på 784 tal bliver til et 28×28-billede.

In [ ]:
raekke = X_traen_np[7]
billede = raekke.reshape(28, 28)
plt.imshow(billede, cmap="gray")
plt.show()

<span style="color:red"><b>LØSNINGSFORSLAG:</b> 28 · 28 = 784 ✓. Netværket ser i øvrigt ALDRIG 28×28-udgaven — det får den flade række på 784 tal og aner ikke, at der "egentlig" er et billede. Tygg lidt på den til opgave 14.8.</span>

##### Opgave 13.3
Er der lige mange af hvert ciffer i træningssættet? Brug `value_counts()` på labels
(pandas-kolonnen `traen_df["label"]`) — og hvorfor er det overhovedet værd at tjekke?
(Tænk på opgave 10.6...)

In [ ]:
traen_df["label"].value_counts().sort_index()

<span style="color:red"><b>LØSNINGSFORSLAG:</b> Fordelingen er pænt jævn (ca. 1500-1700 af hver — i det fulde MNIST er der små forskelle; 1-taller er flest). Det er værd at tjekke, fordi accuracy kun er et ærligt mål, når klasserne er nogenlunde balancerede — ellers gælder Pokémon-lektionen fra opgave 10.6 (den dovne model).</span>

##### Opgave 13.4
Udfyld normaliseringen. Og forklar så (med henvisning til opgave 10.9): hvorfor deler vi
overhovedet med 255 — og hvorfor behøver vi ikke den fulde standardiseringsformel fra
opgave 4.7 her?

In [ ]:
X_alternativ = torch.tensor(X_traen_np / 255.0, dtype=torch.float32)
print(X_alternativ.min().item(), "til", X_alternativ.max().item())

<span style="color:red"><b>LØSNINGSFORSLAG:</b> Rå pixelværdier (0-255) giver kæmpe gradienter og elendig træning (opgave 10.9!). Division med 255 bringer alt ned i [0, 1]. Den fulde (x − mean)/std-formel er unødvendig her, fordi alle 784 features i forvejen ER på samme skala (alle er pixels 0-255) — standardisering pr. kolonne er vigtigst, når kolonnerne har FORSKELLIGE skalaer (HP vs. årsløn).</span>

##### Opgave 13.5 🐛
Koden vil vise et billede, men `reshape` fejler med *"cannot reshape array of size 785
into shape (28,28)"*. 785?! Læs koden — hvilken kolonne har vi glemt at fjerne, og hvad
ville billedet have vist, hvis reshape "bare havde virket"?

In [ ]:
raekke = traen_df.drop(columns=["label"]).values[0]   # ← labelen skal væk først: 785 - 1 = 784
plt.imshow(raekke.reshape(28, 28), cmap="gray")
plt.show()

<span style="color:red"><b>LØSNINGSFORSLAG:</b> <code>traen_df.values</code> tager ALLE 785 kolonner — inklusive labelen, som ikke er en pixel. 785 kan ikke blive til 28×28. Havde den "bare virket", ville cifret stå forskudt med labelen klistret ind som første pixel — et korrumperet billede. Klassisk datafejl: målet gemmer sig i inputtet (det kaldes <i>label leakage</i>, når modellen får facit at se — her ville den bogstaveligt talt kunne AFLÆSE svaret i pixel nr. 1!).</span>

##### Opgave 13.6 ⭐
Koden viser **gennemsnitsbilledet** af alle 3-taller — altså "det gennemsnitlige 3-tal"
(lidt spøgelsesagtigt!). Brug en for-løkke og `plt.subplots` til at vise gennemsnittet af
ALLE 10 cifre i ét grid.

In [ ]:
fig, akser = plt.subplots(2, 5, figsize=(12, 5))
for ciffer, akse in enumerate(akser.ravel()):
    billeder = X_traen_np[y_traen_np == ciffer]
    akse.imshow(billeder.mean(axis=0).reshape(28, 28), cmap="gray")
    akse.set_title(str(ciffer))
    akse.axis("off")
plt.tight_layout()
plt.show()

<span style="color:red"><b>LØSNINGSFORSLAG:</b> Gennemsnitsbillederne er sløret-men-læsbare — det viser, at cifrene ligger nogenlunde samme sted i billedet (ellers ville gennemsnittet være grå grød). Det er i øvrigt en bitte-model i sig selv: man kunne klassificere ved at spørge "hvilket gennemsnitsbillede ligner du mest?" — og det rammer faktisk ~80 %. Jeres netværk skal gøre det bedre!</span>

# 14: Netværket der kan læse

Arkitekturen: **784 pixels ind → 128 skjulte neuroner (ReLU) → 10 point ud** (ét pr.
ciffer). Ingen softmax i modellen — vi bruger `nn.CrossEntropyLoss`, og den vil have de rå
point (reglen fra opgave 12.6!).

Én ny ting: **mini-batches**. I stedet for at vise netværket alle 16.000 billeder før hvert
skridt, viser vi det 64 ad gangen og tager et skridt pr. portion. Det giver mange flere
(og lidt mere støjede) skridt pr. epoke — i praksis træner det både hurtigere og bedre.
Og se: det er bare endnu en for-løkke med slicing, som I har kunnet siden intro-ugen:

In [ ]:
class CifferNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.lag1 = nn.Linear(784, 128)
        self.aktivering = nn.ReLU()
        self.lag2 = nn.Linear(128, 10)    # 10 klasser → 10 rå point (ingen softmax her!)

    def forward(self, x):
        return self.lag2(self.aktivering(self.lag1(x)))

model = CifferNet()
print(model)
print("parametre:", sum(p.numel() for p in model.parameters()))

In [ ]:
model = CifferNet()
tabsfunktion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
batch_stoerrelse = 64

for epoke in range(15):
    for i in range(0, len(X_traen), batch_stoerrelse):      # 64 billeder ad gangen
        X_batch = X_traen[i:i + batch_stoerrelse]
        y_batch = y_traen[i:i + batch_stoerrelse]

        optimizer.zero_grad()                               # rytmen, som altid:
        tab = tabsfunktion(model(X_batch), y_batch)         # nulstil → forward+tab
        tab.backward()                                      # → backward
        optimizer.step()                                    # → step

    print(f"epoke {epoke + 1}: tab på sidste portion = {tab.item():.4f}")

## Eksamen

10 point pr. billede — `argmax(dim=1)` finder indekset med flest point, altså modellens
gæt. Resten kender I:

In [ ]:
with torch.no_grad():
    point = model(X_test)

gaet = point.argmax(dim=1)
accuracy = (gaet == y_test).float().mean()
print(f"test-accuracy: {accuracy.item():.1%}")

Et netværk på ~100.000 parametre, trænet på et par minutter, læser håndskrift med
~95 % sikkerhed. Lad os kigge det i kortene: softmax (NU må vi gerne — vi skal bare *vise*
sandsynligheder, ikke træne) laver de 10 point om til en fordeling:

In [ ]:
i = 0
with torch.no_grad():
    point_i = model(X_test[i:i + 1])
sandsynligheder = torch.softmax(point_i, dim=1).squeeze()

fig, akser = plt.subplots(1, 2, figsize=(10, 3.5))
akser[0].imshow(X_test[i].reshape(28, 28), cmap="gray")
akser[0].set_title(f"label: {y_test[i].item()}")
akser[0].axis("off")
akser[1].bar(range(10), sandsynligheder)
akser[1].set_xticks(range(10))
akser[1].set_title("modellens sandsynligheder")
akser[1].set_xlabel("ciffer")
plt.show()

### Opgaver

##### Opgave 14.1
Hvor meget betyder netværkets størrelse? Træn `CifferNet` med **32**, **128** og **512**
skjulte neuroner (skabelonen tager størrelsen som parameter og tager tid på træningen).
Notér accuracy og træningstid for hver — kan det betale sig at blive større?

In [ ]:
import time

class FleksCifferNet(nn.Module):
    def __init__(self, skjulte):
        super().__init__()
        self.lag1 = nn.Linear(784, skjulte)
        self.aktivering = nn.ReLU()
        self.lag2 = nn.Linear(skjulte, 10)

    def forward(self, x):
        return self.lag2(self.aktivering(self.lag1(x)))

for skjulte in [32, 128, 512]:
    start = time.time()
    model_f = FleksCifferNet(skjulte)
    tabsfunktion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model_f.parameters(), lr=0.001)
    for epoke in range(5):
        for i in range(0, len(X_traen), 64):
            optimizer.zero_grad()
            tab = tabsfunktion(model_f(X_traen[i:i + 64]), y_traen[i:i + 64])
            tab.backward()
            optimizer.step()
    with torch.no_grad():
        acc = (model_f(X_test).argmax(dim=1) == y_test).float().mean()
    print(f"{skjulte:4d} skjulte: accuracy = {acc.item():.1%}, tid = {time.time() - start:.1f} s")

<span style="color:red"><b>LØSNINGSFORSLAG:</b> Typisk (med disse 5 epoker): 32 skjulte ~92 %, 128 ~93 %, 512 ~94 % — og 512 tager omtrent dobbelt så lang tid som 128. Aftagende udbytte: fra 32 til 128 køber man ~1-2 procentpoint, fra 128 til 512 under ét. Størrelse er ikke gratis, og den løser ikke de sidste procent — dét kræver flere epoker (som hovedmodellen med 15) eller en bedre arkitektur (CNN — næste emne!).</span>

##### Opgave 14.2
Udfyld træningsloopets fem manglende linjer — sidste gang, og nu (næsten) uden
hjælpekommentarer. I kan rytmen nu. 💪

In [ ]:
model2 = CifferNet()
tabsfunktion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model2.parameters(), lr=0.001)

for epoke in range(5):
    for i in range(0, len(X_traen), 64):
        X_batch = X_traen[i:i + 64]
        y_batch = y_traen[i:i + 64]
        optimizer.zero_grad()
        tab = tabsfunktion(model2(X_batch), y_batch)
        tab.backward()
        optimizer.step()

with torch.no_grad():
    acc = (model2(X_test).argmax(dim=1) == y_test).float().mean()
print(f"accuracy: {acc.item():.1%}")

<span style="color:red"><b>LØSNINGSFORSLAG:</b> nulstil → forward+tab → backward → step. Hvis I kan skrive de fire linjer i søvne nu, er missionen med dette emne lykkedes.</span>

##### Opgave 14.3
Vis 10 billeder, som modellen gætter FORKERT (skabelonen finder dem — `vis_mnist_billeder`
viser gættet i rødt). Kig på dem: er der nogen, hvor I synes, modellens gæt faktisk er
til at forstå — eller ligefrem bedre end labelen?

In [ ]:
forkerte = (gaet != y_test).nonzero().squeeze()
print("antal fejlgæt:", len(forkerte), "af", len(y_test))

vis_mnist_billeder(X_test[forkerte[:10]], y_test[forkerte[:10]],
                   forudsigelser=gaet[forkerte[:10]], n=10)

<span style="color:red"><b>LØSNINGSFORSLAG:</b> De fleste fejlgæt er genuint grimme tal: 4/9-forvekslinger, 3/5, 7/1... Mange af dem ville mennesker også fejle på (jf. opgave 13.1). En models fejl er tit mere lærerige end dens successer — kig altid på dem!</span>

##### Opgave 14.4
Skift ReLU ud med Sigmoid i `CifferNet` og træn igen (5 epoker). Kan man mærke forskellen
her — og hvorfor er den mindre dramatisk end i 6-lags-eksperimentet fra opgave 12.5?

In [ ]:
class SigmoidCifferNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.lag1 = nn.Linear(784, 128)
        self.aktivering = nn.Sigmoid()
        self.lag2 = nn.Linear(128, 10)

    def forward(self, x):
        return self.lag2(self.aktivering(self.lag1(x)))

model_s = SigmoidCifferNet()
tabsfunktion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model_s.parameters(), lr=0.001)
for epoke in range(5):
    for i in range(0, len(X_traen), 64):
        optimizer.zero_grad()
        tab = tabsfunktion(model_s(X_traen[i:i + 64]), y_traen[i:i + 64])
        tab.backward()
        optimizer.step()

with torch.no_grad():
    acc = (model_s(X_test).argmax(dim=1) == y_test).float().mean()
print(f"accuracy: {acc.item():.1%}")

<span style="color:red"><b>LØSNINGSFORSLAG:</b> Sigmoid-udgaven lander typisk 1-3 procentpoint lavere efter 5 epoker (og indhenter noget med flere epoker). Forskellen er mindre dramatisk end i 12.5, fordi der her kun er ÉT skjult lag — gradienten skal kun gennem én sigmoid-flaskehals, ikke fem. Vanishing gradients er et DYBDE-problem.</span>

##### Opgave 14.5
Udfyld softmax-kaldet og lav sandsynligheds-plottet for et billede, modellen er i TVIVL om
(brug et af fejlgættene fra opgave 14.3 — `forkerte[0]` er et godt bud). Hvordan ser
fordelingen ud sammenlignet med et sikkert gæt?

In [ ]:
i = forkerte[0].item()
with torch.no_grad():
    point_i = model(X_test[i:i + 1])
sandsynligheder = torch.softmax(point_i, dim=1).squeeze()

fig, akser = plt.subplots(1, 2, figsize=(10, 3.5))
akser[0].imshow(X_test[i].reshape(28, 28), cmap="gray")
akser[0].set_title(f"label: {y_test[i].item()}, gæt: {gaet[i].item()}")
akser[0].axis("off")
akser[1].bar(range(10), sandsynligheder)
akser[1].set_xticks(range(10))
plt.show()

<span style="color:red"><b>LØSNINGSFORSLAG:</b> <code>dim=1</code> (klasse-dimensionen — opgave 11.7!). Ved fejlgæt deler sandsynligheden sig typisk mellem 2-3 kandidater (fx 45 % på "9" og 40 % på "4") — modellen VED, den er i tvivl. Den information er guld værd i virkelige systemer: et usikkert svar kan sendes til et menneske i stedet.</span>

##### Opgave 14.6
**Overfitting med egne øjne:** Træn netværket på KUN de første 500 billeder — men i 40
epoker. Mål så accuracy på både træningsbillederne og testsættet. Sæt selv ordet
"overfitting" på det, I ser — og forklar hvad der er sket, med jeres egne ord.

In [ ]:
X_lille = X_traen[:500]
y_lille = y_traen[:500]

model_lille = CifferNet()
tabsfunktion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model_lille.parameters(), lr=0.001)
for epoke in range(40):
    for i in range(0, len(X_lille), 64):
        optimizer.zero_grad()
        tab = tabsfunktion(model_lille(X_lille[i:i + 64]), y_lille[i:i + 64])
        tab.backward()
        optimizer.step()

with torch.no_grad():
    traen_acc = (model_lille(X_lille).argmax(dim=1) == y_lille).float().mean()
    test_acc = (model_lille(X_test).argmax(dim=1) == y_test).float().mean()
print(f"accuracy på de 500 træningsbilleder: {traen_acc.item():.1%}")
print(f"accuracy på testsættet:              {test_acc.item():.1%}")

<span style="color:red"><b>LØSNINGSFORSLAG:</b> Typisk resultat: 100 % på træningsbillederne, men kun ~85-90 % på testsættet. Modellen har lært de 500 billeder UDENAD (den kan dem perfekt!) i stedet for at lære, hvad cifre ER. Gabet mellem trænings- og test-accuracy er selve definitionen på overfitting — og medicinen (mere data, tidligt stop, regularisering, validering) er et hovedtema i næste emne.</span>

##### Opgave 14.7 🐛
En kammerat har "forbedret" CifferNet ved at putte softmax IND i modellen — "så den giver
pæne sandsynligheder". Men træningen er blevet mystisk sløv, og accuracy er faldet. Find
problemet og fjern det (reglen fra opgave 12.6!).

In [ ]:
class KammeratNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.lag1 = nn.Linear(784, 128)
        self.aktivering = nn.ReLU()
        self.lag2 = nn.Linear(128, 10)

    def forward(self, x):
        return self.lag2(self.aktivering(self.lag1(x)))   # rå point — softmaxen er fjernet

model_k = KammeratNet()
tabsfunktion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model_k.parameters(), lr=0.001)
for epoke in range(5):
    for i in range(0, len(X_traen), 64):
        optimizer.zero_grad()
        tab = tabsfunktion(model_k(X_traen[i:i + 64]), y_traen[i:i + 64])
        tab.backward()
        optimizer.step()

with torch.no_grad():
    acc = (model_k(X_test).argmax(dim=1) == y_test).float().mean()
print(f"accuracy: {acc.item():.1%}")

<span style="color:red"><b>LØSNINGSFORSLAG:</b> <code>nn.CrossEntropyLoss</code> kører SELV softmax indeni — med softmax i modellen bliver den kørt TO gange, hvilket maser gradienterne flade og gør træningen sløv og upræcis. Reglen én gang til: rå point til CrossEntropyLoss; softmax kun til VISNING bagefter. Denne fejl er så almindelig i virkeligheden, at den har sin egen FAQ-plads i PyTorch-forummet.</span>

##### Opgave 14.8
Netværket ser kun 784 tal i én lang række — det aner ikke, at pixel 5 og pixel 33 sidder
lige oven over hinanden i billedet. Flyt et ciffer to pixels til højre, og alle 784 tal
ændrer sig — men for os er det "det samme billede".

Hvordan kunne man udnytte, at pixels har NABOER? Brainstorm frit — hvad skulle et smartere
netværk kigge på i stedet for enkelt-pixels?

*Skriv jeres svar her:* $\dots$

<span style="color:red"><b>LØSNINGSFORSLAG:</b> Alle idéer i retning af "kig på små områder/mønstre i stedet for enkelt-pixels" er rigtige: små vinduer, der glider hen over billedet og leder efter streger, buer og hjørner — uanset HVOR i billedet de står. Det er præcis idéen bag <b>konvolutionelle neurale netværk (CNN'er)</b>, som er næste emne på campen. I er klar. 🚀</span>

# EMNET ER FULDFØRT! 🏆

Tag lige et øjeblik og se, hvad I kan nu:

- **Notebook 1:** indlæse, rense, udforske, plotte og standardisere rigtige datasæt (pandas!)
- **Notebook 2:** tensorer, autograd og gradient descent — motoren bag al deep learning
- **Notebook 3:** bygge og træne neurale netværk som klasser, og gennemskue accuracy-fælder
- **Notebook 4:** vælge de rigtige aktiveringsfunktioner — og bevise, at de er nødvendige
- **Notebook 5:** træne et netværk, der læser håndskrift med ~95 % accuracy

Det er ikke legetøj — det er præcis de samme byggesten, som al moderne AI står på. Herfra
handler resten af campen om smartere ARKITEKTURER (CNN'er, transformers...) — men rytmen
`zero_grad → backward → step` følger jer hele vejen.

Er I færdige før tid? `6-Ekstra-opgaver.ipynb` venter med de store, kombinerede
udfordringer. 😈